In [2]:
!pip install langchain chromadb openai tiktoken pypdf langchain_community huggingface-hub langchain_huggingface faiss-cpu wikipedia

In [3]:
# Wikipedia Retrievers


from langchain_community.retrievers import WikipediaRetriever

In [4]:
retriever = WikipediaRetriever(top_k_results = 2, lang='en')

In [8]:
query = ' who is the prime minister of Nepal?'

docs = retriever.invoke(query)


In [9]:
docs

[Document(metadata={'title': 'List of prime ministers of Nepal', 'summary': "The position of the Prime Minister of Nepal (Nepali: नेपालको प्रधानमन्त्री, romanized: Nepālko Pradhānmantrī) in modern form was called by different names at different times of Nepalese history. In the early reign of the Shah dynasty, the Mulkajis (Chief Kajis) or Chautariyas served as prime ministers in a council of 4 Chautariyas, 4 Kajis, and sundry officers. These Bharadars (officers) were drawn from high caste and politically influential families such as the Pande, Basnyat, and Thapa families. The nobility of Gorkha was mainly based from Chhetri families and they had a strong presence in civil administration affairs. \nIn 1804, a single authoritative position of Mukhtiyar was created by Rana Bahadur Shah which carried the executive powers of nation. Mukhtiyar held the position of head of the executive until the adoption of the title of Prime Minister in November 1843 by Mathabar Singh Thapa who became Mukh

In [10]:
for i, doc in enumerate(docs):
  print(f'\n-- Result {i+1} ---')
  print(f'Content:\n{doc.page_content}...')


-- Result 1 ---
Content:
The position of the Prime Minister of Nepal (Nepali: नेपालको प्रधानमन्त्री, romanized: Nepālko Pradhānmantrī) in modern form was called by different names at different times of Nepalese history. In the early reign of the Shah dynasty, the Mulkajis (Chief Kajis) or Chautariyas served as prime ministers in a council of 4 Chautariyas, 4 Kajis, and sundry officers. These Bharadars (officers) were drawn from high caste and politically influential families such as the Pande, Basnyat, and Thapa families. The nobility of Gorkha was mainly based from Chhetri families and they had a strong presence in civil administration affairs. 
In 1804, a single authoritative position of Mukhtiyar was created by Rana Bahadur Shah which carried the executive powers of nation. Mukhtiyar held the position of head of the executive until the adoption of the title of Prime Minister in November 1843 by Mathabar Singh Thapa who became Mukhtiyar as well as Prime Minister and the Chief of the

In [12]:
# Vector store retrievers

from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

embeddings= HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')

In [24]:
documents = [
    Document(page_content="A bunch of scientists bring back dinosaurs and mayhem breaks loose"),
    Document(page_content="Leo DiCaprio gets Mavericked for making the all time best movie"),
    Document(page_content="A psychologist / detective gets lost in a series of dreams within dreams"),
    Document(page_content="A bunch of normal-sized women are supremely wholesome and some men pine after them"),
    Document(page_content="ML includes supervised, unsupervised, semi supervised and reinforcement learning")
]

In [25]:
vectorstore = Chroma.from_documents(
    documents = documents,
    embedding = embeddings,
    collection_name = 'my_collection'
)

In [26]:
retriever =  vectorstore.as_retriever(search_kwargs = {'k':2})

In [27]:
query = " what are types of machine learning ?"

results = retriever.invoke(query)

In [28]:
results

[Document(metadata={}, page_content='ML includes supervised, unsupervised, semi supervised and reinforcement learning'),
 Document(metadata={}, page_content='ML includes supervised, unsupervised, semi supervised and reinforcement learning')]

In [29]:
for i, doc in enumerate(results):
  print(f'\n-- Result {i+1} ---')
  print(f'Content:\n{doc.page_content}...')


-- Result 1 ---
Content:
ML includes supervised, unsupervised, semi supervised and reinforcement learning...

-- Result 2 ---
Content:
ML includes supervised, unsupervised, semi supervised and reinforcement learning...


In [30]:
# Maximal Marginal Relevance (MMR)

docs = [
    Document(page_content="Machine Learning enables computers to learn patterns from data."),
    Document(page_content="Supervised learning trains models using labeled datasets."),
    Document(page_content="Unsupervised learning identifies patterns in data without labels."),
    Document(page_content="Reinforcement learning allows agents to learn by trial and error."),
    Document(page_content="Feature engineering improves model performance by selecting relevant inputs."),
    Document(page_content="Deep Neural Networks are used for complex pattern recognition tasks."),
    Document(page_content="Hyperparameter tuning optimizes model performance on validation data."),
    Document(page_content="Model evaluation measures accuracy and other performance metrics."),
    Document(page_content="Data preprocessing prepares raw data for effective model training."),
    Document(page_content="Machine Learning applications include image recognition and recommendation systems.")
]


In [31]:
from langchain_community.vectorstores import FAISS


vectorstore = FAISS.from_documents(
    documents = docs,
    embedding = embeddings,
)

In [38]:
retriever = vectorstore.as_retriever(
    search_type = 'mmr',                     # activating MMR retriving
    search_kwargs = {'k':3, 'lambda_mult': 0.5} # k = top results , lambda_mult = diversity balance
)

In [39]:
query = ' what is machine learning ?'

result = retriever.invoke(query)

In [40]:
for i, doc in enumerate(result):
  print(f'\n-- Result {i+1} ---')
  print(f'Content:\n{doc.page_content}...')


-- Result 1 ---
Content:
Machine Learning enables computers to learn patterns from data....

-- Result 2 ---
Content:
Model evaluation measures accuracy and other performance metrics....

-- Result 3 ---
Content:
Machine Learning applications include image recognition and recommendation systems....


In [44]:
# Multi query Retriever
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain.retrievers import MultiQueryRetriever

embeddings= HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')

In [46]:

all_docs = [
    Document(
        page_content="Eating a balanced diet rich in fruits, vegetables, and whole grains promotes good health.",
        metadata={'source': 'nutrition_guide', 'author': 'Alice', 'doc_id': 'hl_001', 'category': 'diet'}
    ),
    Document(
        page_content="Regular physical exercise strengthens muscles, improves cardiovascular health, and boosts mood.",
        metadata={'source': 'fitness_manual', 'author': 'Bob', 'doc_id': 'hl_002', 'category': 'exercise'}
    ),
    Document(
        page_content="Drinking plenty of water throughout the day helps maintain hydration and supports metabolism.",
        metadata={'source': 'wellness_tips', 'author': 'Carol', 'doc_id': 'hl_003', 'category': 'hydration'}
    ),
    Document(
        page_content="Getting at least 7–8 hours of quality sleep every night is essential for overall well-being.",
        metadata={'source': 'sleep_guide', 'author': 'David', 'doc_id': 'hl_004', 'category': 'sleep'}
    ),
    Document(
        page_content="Managing stress through meditation, yoga, or deep-breathing exercises can improve mental health.",
        metadata={'source': 'mindfulness_manual', 'author': 'Eve', 'doc_id': 'hl_005', 'category': 'mental_health'}
    ),
    Document(
        page_content="Limiting the intake of processed foods and sugar helps reduce the risk of chronic diseases.",
        metadata={'source': 'nutrition_guide', 'author': 'Alice', 'doc_id': 'hl_006', 'category': 'diet'}
    ),
    Document(
        page_content="Regular health check-ups and preventive screenings can detect issues early and improve outcomes.",
        metadata={'source': 'medical_manual', 'author': 'Frank', 'doc_id': 'hl_007', 'category': 'preventive_care'}
    ),
    Document(
        page_content="Maintaining a healthy weight through proper diet and exercise lowers the risk of obesity-related illnesses.",
        metadata={'source': 'wellness_tips', 'author': 'Carol', 'doc_id': 'hl_008', 'category': 'weight_management'}
    ),
    Document(
        page_content="Avoiding smoking and excessive alcohol consumption supports long-term health.",
        metadata={'source': 'lifestyle_guide', 'author': 'George', 'doc_id': 'hl_009', 'category': 'habits'}
    ),
    Document(
        page_content="Staying socially connected with friends and family contributes to emotional well-being.",
        metadata={'source': 'mindfulness_manual', 'author': 'Eve', 'doc_id': 'hl_010', 'category': 'social_health'}
    ),
     Document(
        page_content="A team of archaeologists discovered a hidden ancient city beneath the desert sands.",
        metadata={'source': 'archaeology_news', 'author': 'Hannah', 'doc_id': 'rand_001', 'category': 'archaeology'}
    ),
    Document(
        page_content="Astronomers detected a new exoplanet that could potentially support life.",
        metadata={'source': 'space_journal', 'author': 'Ian', 'doc_id': 'rand_002', 'category': 'astronomy'}
    ),
    Document(
        page_content="The latest smartphone model features an advanced AI-powered camera and faster processor.",
        metadata={'source': 'tech_review', 'author': 'Jill', 'doc_id': 'rand_003', 'category': 'technology'}
    ),
    Document(
        page_content="A renowned chef shares a secret recipe for a delicious vegan chocolate cake.",
        metadata={'source': 'cooking_blog', 'author': 'Kevin', 'doc_id': 'rand_004', 'category': 'food'}
    ),
    Document(
        page_content="An independent filmmaker released a short movie exploring the concept of time travel.",
        metadata={'source': 'film_magazine', 'author': 'Lena', 'doc_id': 'rand_005', 'category': 'entertainment'}
    )
]


In [47]:
vectorstore = FAISS.from_documents(documents = all_docs, embedding = embeddings)

In [48]:
similarity_retriever = vectorstore.as_retriever(Search_type = 'similarity', search_kwargs = {'k': 5})

In [55]:
from dotenv import load_dotenv
import os

# Load the .env file
load_dotenv('.env')



True

In [69]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
)


multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever = vectorstore.as_retriever(search_kwargs = {'k': 5}),
    llm = ChatHuggingFace(llm=llm),
)

In [70]:
query = "How can I improve my overall health?"

In [71]:
similarity_result = similarity_retriever.invoke(query)
multiquery_result = multiquery_retriever.invoke(query)

In [72]:
for i, doc in enumerate(similarity_result):
  print(f'\n-- Result {i+1} ---')
  print(f'Content:\n{doc.page_content}...')


-- Result 1 ---
Content:
Eating a balanced diet rich in fruits, vegetables, and whole grains promotes good health....

-- Result 2 ---
Content:
Maintaining a healthy weight through proper diet and exercise lowers the risk of obesity-related illnesses....

-- Result 3 ---
Content:
Managing stress through meditation, yoga, or deep-breathing exercises can improve mental health....

-- Result 4 ---
Content:
Regular physical exercise strengthens muscles, improves cardiovascular health, and boosts mood....

-- Result 5 ---
Content:
Limiting the intake of processed foods and sugar helps reduce the risk of chronic diseases....


In [73]:
for i, doc in enumerate(multiquery_result):
  print(f'\n-- Result {i+1} ---')
  print(f'Content:\n{doc.page_content}...')


-- Result 1 ---
Content:
A team of archaeologists discovered a hidden ancient city beneath the desert sands....

-- Result 2 ---
Content:
An independent filmmaker released a short movie exploring the concept of time travel....

-- Result 3 ---
Content:
A renowned chef shares a secret recipe for a delicious vegan chocolate cake....

-- Result 4 ---
Content:
The latest smartphone model features an advanced AI-powered camera and faster processor....

-- Result 5 ---
Content:
Limiting the intake of processed foods and sugar helps reduce the risk of chronic diseases....

-- Result 6 ---
Content:
Maintaining a healthy weight through proper diet and exercise lowers the risk of obesity-related illnesses....

-- Result 7 ---
Content:
Eating a balanced diet rich in fruits, vegetables, and whole grains promotes good health....

-- Result 8 ---
Content:
Avoiding smoking and excessive alcohol consumption supports long-term health....

-- Result 9 ---
Content:
Regular physical exercise strengthens 

In [74]:
# COntextual C0mpression Retriever

from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

In [76]:

mixed_paragraph_docs = [
    Document(
        page_content="Eating a balanced diet rich in vegetables and fruits can improve overall health, and combining it with regular yoga practice helps maintain flexibility and reduces stress.",
        metadata={'doc_id': 'mixp_001', 'author': 'Alice', 'category': 'health_lifestyle'}
    ),
    Document(
        page_content="Astronomers recently discovered an exoplanet that could potentially host life, while at the same time, chefs around the world are experimenting with plant-based ingredients to create innovative recipes.",
        metadata={'doc_id': 'mixp_002', 'author': 'Bob', 'category': 'astronomy_food'}
    ),
    Document(
        page_content="Machine learning algorithms can analyze large datasets to make accurate predictions, and meditation practices help improve focus and mental clarity, showing the power of both technology and mindfulness.",
        metadata={'doc_id': 'mixp_003', 'author': 'Carol', 'category': 'tech_health'}
    )
]

In [77]:
vectorstore = FAISS.from_documents(documents = mixed_paragraph_docs, embedding = embeddings)

In [78]:
base_retriever = vectorstore.as_retriever(search_kwargs = {'k': 5})

In [84]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
)

llm = ChatHuggingFace(llm=llm)

In [85]:
compressor = LLMChainExtractor.from_llm(llm)

In [86]:
compression_retriever= ContextualCompressionRetriever(
    base_compressor = compressor,
    base_retriever = base_retriever
)

In [87]:

query = "what does machine learning algorithm does? "
compressed_result = compression_retriever.invoke(query)

In [88]:
for i, doc in enumerate(compressed_result):
  print(f'\n-- Result {i+1} ---')
  print(f'Content:\n{doc.page_content}...')


-- Result 1 ---
Content:
Machine learning algorithms can analyze large datasets to make accurate predictions....

-- Result 2 ---
Content:
Astronomers recently discovered an exoplanet that could potentially host life, while at the same time, chefs around the world are experimenting with plant-based ingredients to create innovative recipes....
